<a href="https://colab.research.google.com/github/nanda75/RAG-Notebook/blob/main/assignment_2_advanced_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2: Advanced RAG Techniques
## Day 7 - Advanced RAG Fundamentals

**OBJECTIVE:** Implement advanced RAG techniques including postprocessors, response synthesizers, and structured outputs.

**LEARNING GOALS:**
- Understand and implement node postprocessors for filtering and reranking
- Learn different response synthesis strategies (TreeSummarize, Refine)
- Create structured outputs using Pydantic models
- Build advanced retrieval pipelines with multiple processing stages

**DATASET:** Use the same data folder as Assignment 1 (`Day_6/session_2/data/`)

**PREREQUISITES:** Complete Assignment 1 first

**INSTRUCTIONS:**
1. Complete each function by replacing the TODO comments with actual implementation
2. Run each cell after completing the function to test it
3. The answers can be found in the `03_advanced_rag_techniques.ipynb` notebook
4. Each technique builds on the previous one


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q uv
!uv pip install --system -r /content/drive/MyDrive/Outskill/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 58.4 MB/s eta 0:00:00
Using Python 3.13.15 environment at: /usr
Resolved 133 packages in 832ms
Prepared 35 packages in 6.73s
Uninstalled 2 packages in 73ms
Installed 35 packages in 65ms
 + banks==2.5.0
 + colorama==0.4.6
 + dataclasses-json==0.6.7
 + deprecated==1.3.1
 + dirtyjson==1.0.8
 + filetype==1.2.0
 + griffe==2.2.0
 + griffecli==2.2.0
 + griffelib==2.2.0
 + lance-namespace==0.8.6
 + lance-namespace-urllib3-client==0.8.6
 + lancedb==0.37.1
 + llama-index==0.14.24
 + llama-index-core==0.14.24
 + llama-index-embeddings-huggingface==0.7.0
 + llama-index-embeddings-openai==0.6.0
 + llama-index-instrumentation==0.6.0
 + llama-index-llms-huggingface-api==0.7.0
 + llama-index-llms-openai==0.7.10
 + llama-index-llms-openai-like==0.7.2
 + llama-index-llms-openrouter==0.5.1
 + llama-index-readers-file==0.6.0
 + llama-index-vector-stores-lancedb==0.5.0
 + llama-index-workflows==2.23.3
 + marshmallow==3.26.2
 + mypy-extensions==1.1.0


In [4]:
import os
from getpass import getpass

# securely input your key
os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter key")
print("✓ OpenRouter key set successfully")

Enter your OpenRouter key··········
✓ OpenRouter key set successfully


In [5]:
# Import required libraries for advanced RAG
import os
from pathlib import Path
from typing import Dict, List, Optional, Any
from pydantic import BaseModel, Field

# Core LlamaIndex components
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext, Settings
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import VectorIndexRetriever

# Vector store
from llama_index.vector_stores.lancedb import LanceDBVectorStore

# Embeddings and LLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openrouter import OpenRouter

# Advanced RAG components (we'll use these in the assignments)
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.response_synthesizers import TreeSummarize, Refine, CompactAndRefine
from llama_index.core.output_parsers import PydanticOutputParser

print("Advanced RAG libraries imported successfully!")


Advanced RAG libraries imported successfully!


In [6]:
# Configure Advanced RAG Settings (Using OpenRouter)
def setup_advanced_rag_settings():
    """
    Configure LlamaIndex with optimized settings for advanced RAG.
    Uses local embeddings and OpenRouter for LLM operations.
    """
    # Check for OpenRouter API key
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        print("OPENROUTER_API_KEY not found - LLM operations will be limited")
        print("You can still complete postprocessor and retrieval exercises")
    else:
        print("OPENROUTER_API_KEY found - full advanced RAG functionality available")

        # Configure OpenRouter LLM
        Settings.llm = OpenRouter(
            api_key=api_key,
            model="gpt-4o",
            temperature=0.1  # Lower temperature for more consistent responses
        )

    # Configure local embeddings (no API key required)
    Settings.embed_model = HuggingFaceEmbedding(
        model_name="BAAI/bge-small-en-v1.5",
        trust_remote_code=True
    )

    # Advanced RAG configuration
    Settings.chunk_size = 512  # Smaller chunks for better precision
    Settings.chunk_overlap = 50

    print("Advanced RAG settings configured")
    print("- Chunk size: 512 (optimized for precision)")
    print("- Using local embeddings for cost efficiency")
    print("- OpenRouter LLM ready for response synthesis")

# Setup the configuration
setup_advanced_rag_settings()


OPENROUTER_API_KEY found - full advanced RAG functionality available


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Advanced RAG settings configured
- Chunk size: 512 (optimized for precision)
- Using local embeddings for cost efficiency
- OpenRouter LLM ready for response synthesis


In [8]:
# Setup: Create index from Assignment 1 (reuse the basic functionality)
def setup_basic_index(data_folder: str, force_rebuild: bool = False):
    """
    Create a basic vector index that we'll enhance with advanced techniques.
    This reuses the concepts from Assignment 1.
    """
    # Create vector store
    vector_store = LanceDBVectorStore(
        uri="/content/storage/advanced_rag_vectordb",
        table_name="documents"
    )

    # Load documents
    if not Path(data_folder).exists():
        print(f"Data folder not found: {data_folder}")
        return None

    reader = SimpleDirectoryReader(input_dir=data_folder, recursive=True)
    documents = reader.load_data()

    # Create storage context and index
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True
    )

    print(f"Basic index created with {len(documents)} documents")
    print("Ready for advanced RAG techniques!")
    return index

data_folder_path = "/content/drive/MyDrive/Outskill/data"
# Create the basic index
print("Setting up basic index for advanced RAG...")
index = setup_basic_index(data_folder=data_folder_path)

if index:
    print("Ready to implement advanced RAG techniques!")
else:
    print("Failed to create index - check data folder path")


Setting up basic index for advanced RAG...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/90 [00:00<?, ?it/s]

Basic index created with 42 documents
Ready for advanced RAG techniques!
Ready to implement advanced RAG techniques!


## 1. Node Postprocessors - Similarity Filtering

**Concept:** Postprocessors refine retrieval results after the initial vector search. The `SimilarityPostprocessor` filters out chunks that fall below a relevance threshold.

**Why it matters:** Raw vector search often returns some irrelevant results. Filtering improves precision and response quality.

Complete the function below to create a query engine with similarity filtering.


In [9]:
def create_query_engine_with_similarity_filter(index, similarity_cutoff: float = 0.3, top_k: int = 10):
    """
    Create a query engine that filters results based on similarity scores.

    TODO: Complete this function to create a query engine with similarity postprocessing.


    Args:
        index: Vector index to query
        similarity_cutoff: Minimum similarity score (0.0 to 1.0)
        top_k: Number of initial results to retrieve before filtering

    Returns:
        Query engine with similarity filtering
    """
    # HINT: Use index.as_query_engine() with node_postprocessors parameter containing SimilarityPostprocessor
    # TODO: Create similarity postprocessor with the cutoff threshold
    similarity_processor = SimilarityPostprocessor(
        similarity_cutoff=similarity_cutoff
    )

    # TODO: Create query engine with similarity filtering
    query_engine = index.as_query_engine(
        similarity_top_k=top_k,
        node_postprocessors=[
            SimilarityPostprocessor(similarity_cutoff=0.2),  # First filter by relevance
        ]
    )

    return query_engine

# Test the function
if index:
    filtered_engine = create_query_engine_with_similarity_filter(index, similarity_cutoff=0.3)

    if filtered_engine:
        print("Query engine with similarity filtering created")

        # Test query
        test_query = "What are the benefits of AI agents?"
        print(f"\nTesting query: '{test_query}'")

        # Uncomment when implemented:
        response = filtered_engine.query(test_query)
        print(f"Response: {response}")
        print(f"Filtered Response: {response}")
    else:
        print("Failed to create filtered query engine")
else:
    print("No index available - run previous cells first")

Query engine with similarity filtering created

Testing query: 'What are the benefits of AI agents?'
Response: AI agents provide numerous advantages, such as advanced reasoning and planning skills that enable them to tackle complex challenges and make independent decisions. They can perform tasks either independently or in collaboration with human users, thereby enhancing efficiency and effectiveness. Multi-agent systems are particularly adept at managing parallel tasks and offering varied feedback, which is advantageous for collaborative tasks. Furthermore, AI agents can be tailored to specific tasks, ensuring that the most appropriate agents are engaged in planning and execution. Their adaptability and capacity to modify plans based on new information enhance their strong performance across diverse applications.
Filtered Response: AI agents provide numerous advantages, such as advanced reasoning and planning skills that enable them to tackle complex challenges and make independent de

## 2. Structured Outputs with Pydantic Models

**Concept:** Structured outputs ensure predictable, parseable responses using Pydantic models. This is essential for API endpoints and data pipelines.

**Why it matters:** Instead of free-text responses, you get type-safe, validated data structures that applications can reliably process.

Complete the function below to create a structured output system for extracting research paper information.


In [12]:
# First, define the Pydantic models for structured outputs
class ResearchPaperInfo(BaseModel):
    """Structured information about a research paper or AI concept."""
    title: str = Field(description="The main title or concept name")
    key_points: List[str] = Field(description="3-5 main points or findings")
    applications: List[str] = Field(description="Practical applications or use cases")
    summary: str = Field(description="Brief 2-3 sentence summary")


# Import the missing component
from llama_index.core.program import LLMTextCompletionProgram
from llama_index.core.output_parsers import PydanticOutputParser

def create_structured_output_program(output_model: BaseModel = ResearchPaperInfo):
    """
    Create a structured output program using Pydantic models.

    TODO: Complete this function to create a structured output program.

    Args:
        output_model: Pydantic model class for structured output

    Returns:
        LLMTextCompletionProgram that returns structured data
    """
    # TODO: Create output parser with the Pydantic model (HINT: Use PydanticOutputParser)
    output_parser = PydanticOutputParser(ResearchPaperInfo)

    # TODO: Create the structured output program (HINT output_parser and prompt_template_str will be passed to the function)
    #program = LLMTextCompletionProgram.from_defaults(?)
    program = LLMTextCompletionProgram.from_defaults(
    output_parser=output_parser,
    prompt_template_str="Extract Structured information about a research paper or AI concept.:\n"
                        "{context}\n\n"
                        "Query: {query}\n\n"
                        "Provide the recipe information in the specified JSON format.",
    verbose=True
    )
    return program

    # PLACEHOLDER - Replace with actual implementation
    print(f"TODO: Create structured output program with {output_model.__name__}")
    return None

# Test the function
if index:
    structured_program = create_structured_output_program(ResearchPaperInfo)

    if structured_program:
        print("Structured output program created")

        # Test with retrieval and structured extraction
        structure_query = "Tell me about AI agents and their capabilities"
        print(f"Testing structured query: '{structure_query}'")

        # Get context for structured extraction (Uncomment when implemented)
        retriever = VectorIndexRetriever(index=index, similarity_top_k=3)
        nodes = retriever.retrieve(structure_query)
        context = "\n".join([node.text for node in nodes])

        # Uncomment when implemented:
        response = structured_program(context=context, query=structure_query)
        print(f"Structured Response:\n{response}")
        print("   (Complete the function above to get structured JSON output)")

        print("Expected output format:")
        print(f"   - title: {response.title}")
        print(f"   - key_points: {response.key_points}")
        print(f"   - applications: {response.applications}")
        print(f"   - summary: {response.summary}")
    else:
        print("Failed to create structured output program")
else:
    print("No index available - run previous cells first")


Structured output program created
Testing structured query: 'Tell me about AI agents and their capabilities'
Structured Response:
title='Agentic AI Systems' key_points=['Agentic Design Patterns provide architectural templates for creating autonomous agents.', 'Multi-Agent Scaling Laws describe quantitative relationships between agent count and system performance.', 'Verbal Reinforcement Learning uses language feedback for reinforcement rather than numeric rewards.', 'Dynamic teams of agents can be more effective by bringing agents in and out based on need.', 'Challenges exist in evaluating agent systems due to varying benchmarks and potential biases.'] applications=['Rapid development and deployment of robust agentic AI systems in finance.', 'Improving AI-driven agents for real-world applicability.', 'Mitigating harmful language model biases in AI agents.'] summary='This research paper surveys the progression from static language models to dynamic, autonomous agents, highlighting key t